# TTZ YOLOv8n v3 Fine-tune
## 推筒子水平牌偵測訓練

### 前置：上傳 dataset
1. 下載 `mahjong_v3_dataset.zip` 到本機
2. 執行左邊 cell 上傳

或手動上傳到 Colab：左側檔案區 → 上傳

In [ ]:
# @title 1. 上傳 dataset zip（或跳過已上傳）
import os, zipfile, glob

ZIP_PATH = '/content/mahjong_v3_dataset.zip'
DST = '/content/mahjong_v3_dataset'

if not os.path.exists(DST):
    from google.colab import files
    uploaded = files.upload()
    # 取第一個上傳的 zip
    for fn in uploaded:
        if fn.endswith('.zip'):
            os.rename(fn, ZIP_PATH)
            break
    print(f'Extracting {ZIP_PATH}...')
    with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
        zf.extractall('/content/')
    print('Done')
else:
    print('Dataset already exists')

In [ ]:
# @title 2. 驗證 dataset
train_imgs = glob.glob(f'{DST}/images/train/*.jpg')
val_imgs = glob.glob(f'{DST}/images/val/*.jpg')
train_lbls = glob.glob(f'{DST}/labels/train/*.txt')
val_lbls = glob.glob(f'{DST}/labels/val/*.txt')
print(f'Train: {len(train_imgs)} imgs, {len(train_lbls)} labels')
print(f'Val:   {len(val_imgs)} imgs, {len(val_lbls)} labels')
!cat {DST}/data.yaml

In [ ]:
# @title 3. 安裝 ultralytics
!pip install -q ultralytics
import torch
print(f'GPU: {torch.cuda.get_device_name(0)}' if torch.cuda.is_available() else 'WARNING: No GPU!')

In [ ]:
# @title 4. 開始訓練
from ultralytics import YOLO

model = YOLO('yolov8n.pt')  # 從頭預訓練

results = model.train(
    data=f'{DST}/data.yaml',
    epochs=200,
    patience=30,
    batch=32,
    imgsz=640,
    device='0',
    project='/content',
    name='yolov8n_v3',
    optimizer='AdamW',
    lr0=0.005,
    lrf=0.01,
    warmup_epochs=3,
    degrees=5,
    translate=0.1,
    scale=0.5,
    shear=2,
    flipud=0.0,
    fliplr=0.5,
    mosaic=0.5,
    hsv_h=0.015,
    hsv_s=0.4,
    hsv_v=0.3,
    val=True,
    plots=True,
    save=True,
    save_period=10,
    amp=True,
)
print('Training complete!')

In [ ]:
# @title 5. 查看結果
import pandas as pd
csv_path = '/content/yolov8n_v3/results.csv'
if os.path.exists(csv_path):
    df = pd.read_csv(csv_path)
    best_idx = df['metrics/mAP50(B)'].idxmax()
    best = df.iloc[best_idx]
    print(f"Best epoch: {int(best['epoch'])}")
    print(f"mAP50: {best['metrics/mAP50(B)']:.4f}")
    print(f"mAP50-95: {best['metrics/mAP50-95(B)']:.4f}")
    print(f"Precision: {best['metrics/precision(B)']:.4f}")
    print(f"Recall: {best['metrics/recall(B)']:.4f}")
else:
    print('Training not finished yet')

In [ ]:
# @title 6. 匯出 ONNX (imgsz=160 給 Pi)
best_pt = '/content/yolov8n_v3/weights/best.pt'
if os.path.exists(best_pt):
    model = YOLO(best_pt)
    model.export(format='onnx', imgsz=160, half=True, simplify=True)
    !cp /content/yolov8n_v3/weights/best_160.onnx /content/ttz_v3_160.onnx
    !cp /content/yolov8n_v3/weights/best.pt /content/ttz_v3.pt
    print('ONNX exported!')
    !ls -lh /content/ttz_v3_160.onnx
else:
    print('best.pt not found')

In [ ]:
# @title 7. 下載模型
from google.colab import files
files.download('/content/ttz_v3_160.onnx')
files.download('/content/ttz_v3.pt')